In [1]:
import numpy as np
import torch
import pickle as pkl
from itertools import permutations
import matplotlib.pyplot as plt
from tqdm import tqdm 
import sys
import math
sys.path.append("./gpt2_model/")
from gpt2 import GPT

In [2]:
gpu_id=1
device = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"

In [3]:
n_states = 6
block_size = 256          # must match what the model was trained with

In [4]:
model = pkl.load(open(f"pcsk9_gpt_model_best_{n_states}_state.pkl", "rb")).to(device)
model.eval()

GPT(
  (token_embedding): Embedding(6, 480)
  (position_embedding): Embedding(256, 480)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((480,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x Head(
            (key): Linear(in_features=480, out_features=60, bias=False)
            (query): Linear(in_features=480, out_features=60, bias=False)
            (value): Linear(in_features=480, out_features=60, bias=False)
          )
        )
        (proj): Linear(in_features=480, out_features=480, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ln2): LayerNorm((480,), eps=1e-05, elementwise_affine=True)
      (ff): FFNN(
        (net): Sequential(
          (0): Linear(in_features=480, out_features=1920, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1920, out_features=480, bias=True)
          (3): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (1

In [5]:
def generate_extended_ensemble(model, true_trajs, block_size, device,
                                 length_factor=2, batch_cap=64):
    """
    Returns a list of generated trajectories (np arrays), one per usable
    true trajectory (those with len > block_size). Each generated
    trajectory is `length_factor` times the length of its corresponding
    true trajectory, seeded on that true trajectory's first `block_size`
    states. Batches the generation for speed since GPT.generate accepts
    a batch dimension.
    """
    usable = [t for t in true_trajs if len(t) > block_size]
    skipped = len(true_trajs) - len(usable)
    if skipped:
        print(f"Skipping {skipped} trajectories too short to seed "
              f"(need > {block_size} steps).")

    generated = []
    n_batches = math.ceil(len(usable) / batch_cap)

    for start in tqdm(range(0, len(usable), batch_cap),
                       total=n_batches,
                       desc="Generating trajectories",
                       unit="batch"):
        batch = usable[start:start + batch_cap]
        seeds = torch.stack(
            [torch.tensor(t[:block_size], dtype=torch.long) for t in batch]
        ).to(device)

        # target length for each trajectory in this batch = n * true length
        needed = [int(len(t) * length_factor) for t in batch]
        n_new = max(needed)   # generate enough for the longest target in batch

        with torch.no_grad():
            out = model.generate(seeds, max_new_tokens=n_new).cpu().numpy()

        for i, need in enumerate(needed):
            generated.append(out[i, block_size:block_size + need])

    return generated

In [6]:
true_trajs=pkl.load(open(f"/path/to/pcsk9_gmm_nstates_{n_states}.pkl", "rb"))

In [7]:
#length_factors = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]#, 2.0, 3.0, 5.0, 7.0, 10.0]
length_factors = [1.0,]

for length_factor in length_factors:
    generated_trajs = generate_extended_ensemble(
        model, true_trajs, block_size, device, length_factor=length_factor
    )

    gen_traj_concat = np.concatenate(generated_trajs)
    print(f"\nlength_factor={length_factor}: total generated steps = {len(gen_traj_concat)}")

    counts = np.bincount(gen_traj_concat, minlength=n_states)
    population = counts / counts.sum() * 100
    for state, pop in enumerate(population):
        print(f"  State {state + 1}: {pop:.2f}%")

    lf_tag = f"{round(length_factor * 10):04d}"
    out_path = (
        f"generated_extended_length_{lf_tag}_pcsk9_gpt_model_best_"
        f"{n_states}_state_block_size_{block_size}.pkl"
    )
    with open(out_path, "wb") as f:
        pkl.dump(generated_trajs, f)

Skipping 1 trajectories too short to seed (need > 256 steps).


Generating trajectories: 100%|████████████████████████████████████████████████████████████████████████████████████████| 7/7 [59:05<00:00, 506.55s/batch]


length_factor=1.0: total generated steps = 1912878
  State 1: 18.99%
  State 2: 27.20%
  State 3: 12.45%
  State 4: 11.41%
  State 5: 5.04%
  State 6: 24.91%
